In [ ]:
# ============================================================
# 1. GET ACTIVE SNOWFLAKE SESSION
# ============================================================

from snowflake.snowpark.context import get_active_session

session = get_active_session()

print("Snowflake session connected")


# ============================================================
# 2. CONFIGURE DATABASE, SCHEMA AND WAREHOUSE
# ============================================================

session.sql("USE DATABASE ML_PROJECT").collect()
session.sql("USE SCHEMA INCOME_PREDICTION").collect()
session.sql("USE WAREHOUSE ML_WH").collect()

print("Database  : ML_PROJECT")
print("Schema    : INCOME_PREDICTION")
print("Warehouse : ML_WH")


# ============================================================
# 3. LOAD DATA FROM SNOWFLAKE
# ============================================================

df = session.table("RAW_ADULT_DATA")

pdf = df.to_pandas()

print("\nDataset loaded successfully")
print("Shape:", pdf.shape)

print("\nFirst 5 rows:")
display(pdf.head())


# ============================================================
# 4. BASIC DATA CLEANING
# ============================================================

# Remove spaces from string columns
string_columns = pdf.select_dtypes(include=["object"]).columns

for column in string_columns:
    pdf[column] = pdf[column].str.strip()

# Replace ? with missing value
pdf = pdf.replace("?", None)

# Remove rows where target is missing
pdf = pdf.dropna(subset=["INCOME"])

print("\nAfter cleaning:")
print("Shape:", pdf.shape)


# ============================================================
# 5. DEFINE FEATURES AND TARGET
# ============================================================

X = pdf.drop("INCOME", axis=1)

y = pdf["INCOME"]

print("\nTarget distribution:")
print(y.value_counts())


# ============================================================
# 6. DEFINE CATEGORICAL AND NUMERIC FEATURES
# ============================================================

categorical = [
    "WORKCLASS",
    "EDUCATION",
    "MARITAL_STATUS",
    "OCCUPATION",
    "RELATIONSHIP",
    "RACE",
    "SEX",
    "NATIVE_COUNTRY"
]

numeric = [
    "AGE",
    "FNLWGT",
    "EDUCATION_NUM",
    "CAPITAL_GAIN",
    "CAPITAL_LOSS",
    "HOURS_PER_WEEK"
]


# ============================================================
# 7. IMPORT MACHINE LEARNING LIBRARIES
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)


# ============================================================
# 8. PREPROCESSING
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical
        )
    ],
    remainder="passthrough"
)


# ============================================================
# 9. CREATE RANDOM FOREST MODEL
# ============================================================

model = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "classifier",
        RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        )
    )
])


# ============================================================
# 10. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining samples:", len(X_train))
print("Testing samples :", len(X_test))


# ============================================================
# 11. TRAIN MODEL
# ============================================================

print("\nTraining Random Forest model...")

model.fit(X_train, y_train)

print("Model training completed")


# ============================================================
# 12. MAKE PREDICTIONS
# ============================================================

predictions = model.predict(X_test)


# ============================================================
# 13. MODEL EVALUATION
# ============================================================

accuracy = accuracy_score(
    y_test,
    predictions
)

print("\n====================================")
print("MODEL EVALUATION")
print("====================================")

print("Accuracy:", round(accuracy, 4))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        predictions
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        predictions
    )
)


# ============================================================
# 14. SAVE MODEL
# ============================================================

import joblib

model_path = "/tmp/income_model.pkl"

joblib.dump(
    model,
    model_path
)

print("\nModel saved to:")
print(model_path)


# ============================================================
# 15. TEST THE MODEL WITH ONE SAMPLE
# ============================================================

sample = X_test.iloc[[0]]

sample_prediction = model.predict(sample)[0]

print("\n====================================")
print("SAMPLE PREDICTION")
print("====================================")

print("Predicted Income:", sample_prediction)
print("Actual Income   :", y_test.iloc[0])


# ============================================================
# 16. FINAL STATUS
# ============================================================

print("\n====================================")
print("PROJECT STATUS")
print("====================================")

print("✓ Snowflake connected")
print("✓ Database configured")
print("✓ Schema configured")
print("✓ Warehouse configured")
print("✓ Dataset loaded")
print("✓ Data cleaned")
print("✓ Features prepared")
print("✓ Random Forest trained")
print("✓ Model evaluated")
print("✓ Model saved")
print("✓ Prediction tested")

In [ ]:
# ============================================================
# TRAIN INCOME PREDICTION MODEL
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Make sure pandas dataframe exists
pdf = df.to_pandas()

# ------------------------------------------------------------
# 1. CLEAN DATA
# ------------------------------------------------------------

# Remove whitespace from text columns
text_columns = pdf.select_dtypes(include=["object"]).columns

for col in text_columns:
    pdf[col] = pdf[col].str.strip()

# Convert ? to missing values
pdf = pdf.replace("?", None)

# Remove rows without target
pdf = pdf.dropna(subset=["INCOME"])

print("Cleaned dataset shape:", pdf.shape)


# ------------------------------------------------------------
# 2. FEATURES AND TARGET
# ------------------------------------------------------------

X = pdf.drop("INCOME", axis=1)
y = pdf["INCOME"]

print("\nTarget distribution:")
print(y.value_counts())


# ------------------------------------------------------------
# 3. FEATURE TYPES
# ------------------------------------------------------------

categorical_features = [
    "WORKCLASS",
    "EDUCATION",
    "MARITAL_STATUS",
    "OCCUPATION",
    "RELATIONSHIP",
    "RACE",
    "SEX",
    "NATIVE_COUNTRY"
]

numeric_features = [
    "AGE",
    "FNLWGT",
    "EDUCATION_NUM",
    "CAPITAL_GAIN",
    "CAPITAL_LOSS",
    "HOURS_PER_WEEK"
]


# ------------------------------------------------------------
# 4. PREPROCESSING
# ------------------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)


# ------------------------------------------------------------
# 5. RANDOM FOREST
# ------------------------------------------------------------

model = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "classifier",
        RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        )
    )
])


# ------------------------------------------------------------
# 6. TRAIN / TEST SPLIT
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining rows:", len(X_train))
print("Testing rows :", len(X_test))


# ------------------------------------------------------------
# 7. TRAIN
# ------------------------------------------------------------

print("\nTraining Random Forest...")

model.fit(X_train, y_train)

print("Training completed successfully!")


# ------------------------------------------------------------
# 8. PREDICTION
# ------------------------------------------------------------

predictions = model.predict(X_test)


# ------------------------------------------------------------
# 9. EVALUATION
# ------------------------------------------------------------

accuracy = accuracy_score(
    y_test,
    predictions
)

print("\n========================================")
print("MODEL EVALUATION")
print("========================================")

print("Accuracy:", round(accuracy, 4))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        predictions
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        predictions
    )
)


# ------------------------------------------------------------
# 10. SAVE MODEL
# ------------------------------------------------------------

import joblib

model_path = "/tmp/income_model.pkl"

joblib.dump(
    model,
    model_path
)

print("\n========================================")
print("MODEL SAVED")
print("========================================")

print(model_path)


# ------------------------------------------------------------
# 11. TEST ONE PREDICTION
# ------------------------------------------------------------

sample = X_test.iloc[[0]]

predicted_income = model.predict(sample)[0]
actual_income = y_test.iloc[0]

print("\n========================================")
print("SAMPLE PREDICTION")
print("========================================")

print("Predicted:", predicted_income)
print("Actual   :", actual_income)

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

# Create the stage
session.sql("""
CREATE STAGE IF NOT EXISTS ML_PROJECT.INCOME_PREDICTION.MODEL_STAGE
""").collect()

print("MODEL_STAGE created")

# Upload the trained model
result = session.file.put(
    "/tmp/income_model.pkl",
    "@ML_PROJECT.INCOME_PREDICTION.MODEL_STAGE",
    auto_compress=False,
    overwrite=True
)

print("Upload result:")
for r in result:
    print(r)

# Verify
files = session.file.list(
    "@ML_PROJECT.INCOME_PREDICTION.MODEL_STAGE"
)

print("\nFiles in stage:")
for f in files:
    print(f.name)

print("\nMODEL UPLOAD COMPLETE")